# Exploratory Data Analysis (EDA)

This notebook performs EDA on cleaned Amazon review data and summarizes key insights.

In [ ]:
# ==========================================
# 1. Import Libraries
# ==========================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set(style="whitegrid")

In [ ]:
# ==========================================
# 2. Load Cleaned Data
# ==========================================
df = pd.read_csv("../data/processed/cleaned_reviews.csv")

print("Shape:", df.shape)
df.head()

## Data Quality Snapshot

Quick checks to understand missing values and duplicate rows before deeper analysis.

In [ ]:
# ==========================================
# Data Quality Snapshot
# ==========================================
missing_values = df.isnull().sum().sort_values(ascending=False)
print("Top columns by missing values:\n", missing_values.head(10))
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# ==========================================
# 3. Basic Overview
# ==========================================
df.info()
df.describe()

## 1. Rating Distribution

**Insight**
- Check if most ratings are 4-5.
- Identify presence of low ratings (1-2).

In [ ]:
# ==========================================
# Rating Distribution
# ==========================================
plt.figure(figsize=(8, 4))
sns.countplot(x="rating", data=df, order=sorted(df["rating"].dropna().unique()), palette="Blues")
plt.title("Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 2. Sentiment Distribution

**Insight**
- Compare Positive vs Negative vs Neutral.
- Validate sentiment logic.

In [ ]:
# ==========================================
# Sentiment Distribution
# ==========================================
plt.figure(figsize=(8, 4))
sns.countplot(x="sentiment", data=df, order=["Negative", "Neutral", "Positive"], palette="viridis")
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 3. Verified vs Rating

**Insight**
- Do verified users give lower/more realistic ratings?

In [ ]:
# ==========================================
# Verified vs Rating
# ==========================================
plt.figure()
sns.boxplot(x="verified", y="rating", data=df)
plt.title("Verified vs Rating")
plt.xlabel("Verified Purchase")
plt.ylabel("Rating")
plt.show()

## 4. Product-Level Analysis (ASIN)

**Insight**
- Identify consistently poor-performing products.
- Highlight top-performing products.

In [ ]:
# ==========================================
# Product Performance
# ==========================================
product_stats = df.groupby("asin").agg(avg_rating=("rating", "mean"), review_count=("rating", "count"))
product_stats = product_stats[product_stats["review_count"] >= 10].sort_values("avg_rating")

print("Worst Products (min 10 reviews):\n", product_stats.head(5))
print("\nBest Products (min 10 reviews):\n", product_stats.tail(5))

# Visualize Top & Bottom Products with minimum-review filter
plt.figure(figsize=(10, 5))
product_stats.head(5)["avg_rating"].plot(kind="barh", title="Worst Products (Lowest Avg Rating)")
plt.xlabel("Average Rating")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
product_stats.tail(5)["avg_rating"].plot(kind="barh", title="Best Products (Highest Avg Rating)")
plt.xlabel("Average Rating")
plt.tight_layout()
plt.show()

## 5. Review Length vs Rating

**Insight**
- Negative reviews are often longer (complaints).

In [ ]:
# ==========================================
# Review Length vs Rating
# ==========================================
plt.figure()
sns.boxplot(x="rating", y="review_length", data=df)
plt.title("Review Length vs Rating")
plt.xlabel("Rating")
plt.ylabel("Review Length")
plt.show()

## 6. Helpful Votes Analysis

**Insight**
- Are low ratings getting more helpful votes?

In [ ]:
# ==========================================
# Helpful Votes vs Rating
# ==========================================
plt.figure(figsize=(8, 4))
sns.scatterplot(x="rating", y="helpful", data=df, alpha=0.6)
plt.title("Helpful Votes vs Rating")
plt.xlabel("Rating")
plt.ylabel("Helpful Votes")
plt.tight_layout()
plt.show()

## 7. Correlation Matrix

**Insight**
- Relationship between `rating`, `helpful`, and `review_length`.

In [ ]:
# ==========================================
# Correlation Matrix
# ==========================================
plt.figure()
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

## 8. Word Frequency Analysis

**Insight**
- Positive words often include terms like `love`, `great`, `perfect`.
- Negative words often include terms like `poor`, `useless`, `cheap`.

In [ ]:
# ==========================================
# Word Frequency (Overall)
# ==========================================
words = " ".join(df["cleaned_text"]).split()
common_words = Counter(words).most_common(20)

print("Top 20 Words:\n", common_words)

# ==========================================
# Positive Reviews Words
# ==========================================
positive_words = " ".join(df[df["sentiment"] == "Positive"]["cleaned_text"]).split()
negative_words = " ".join(df[df["sentiment"] == "Negative"]["cleaned_text"]).split()

print("Top Positive Words:\n", Counter(positive_words).most_common(10))
print("\nTop Negative Words:\n", Counter(negative_words).most_common(10))

## Word Frequency Visualization

Visual comparison of the most common terms in positive and negative reviews.

In [ ]:
# ==========================================
# Word Frequency Visualization
# ==========================================
positive_top = pd.DataFrame(Counter(positive_words).most_common(10), columns=["word", "count"])
negative_top = pd.DataFrame(Counter(negative_words).most_common(10), columns=["word", "count"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=positive_top, x="count", y="word", ax=axes[0], palette="Greens")
axes[0].set_title("Top Positive Words")
axes[0].set_xlabel("Frequency")
axes[0].set_ylabel("Word")

sns.barplot(data=negative_top, x="count", y="word", ax=axes[1], palette="Reds")
axes[1].set_title("Top Negative Words")
axes[1].set_xlabel("Frequency")
axes[1].set_ylabel("Word")

plt.tight_layout()
plt.show()

## Final Summary

The following summary consolidates the major findings from this EDA.

In [ ]:
# ==========================================
# Key Insights Summary
# ==========================================

print("""
KEY INSIGHTS:

1. Majority of reviews are positive (4-5 ratings dominate)
2. Negative reviews highlight product quality issues
3. Verified users tend to give more reliable ratings
4. Certain products consistently underperform
5. Longer reviews are usually associated with complaints
6. Negative reviews often receive more helpful votes
""")